# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankitpaul6201/Fly-rank-intern-01/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Plain-Words Contract (Five Core Answers):

1. **Unit of Analysis (Grain):** **One row = One single pseudonymized content item (`content_id`)** for a specific client site (`client_id`).
2. **Table(s) Used:** `data/raw/content_refresh_anonymized.csv` (and warehouse table `fact_content_daily_performance` for mid-panel month `month=2026-03`).
3. **Time Window:** 90-day baseline snapshot window with trailing 30-day performance comparison windows (`impressions_last_30d` vs `impressions_prev_30d`).
4. **Target / Proxy Predicted:** Observed relative impression drop proxy $\Delta \text{Imp}_{\text{rel}} = \frac{\text{impressions}_{\text{last 30d}} - \text{impressions}_{\text{prev 30d}}}{\text{impressions}_{\text{prev 30d}} + 1} \times 100$, producing a binary decay flag ($\Delta \text{Imp}_{\text{rel}} < -15.0\%$) to rank top-50 refresh candidates.
5. **Deliberately Excluded:** `trend_direction` and `trend_pct` (target leakage columns derived directly from the outcome window) and `health_score` / `is_declining_label` (circular internal product tags).

In [1]:
# Section 1 Code: Dataset Loading & Contract Specification Summary
import pandas as pd
import numpy as np
import os

data_path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    data_path = "../../data/raw/content_refresh_anonymized.csv"

df_raw = pd.read_csv(data_path)

print("=== DATA CONTRACT SPECIFICATION SUMMARY ===")
print(f"1. Grain           : One row = 1 content_id (pseudonymized content page)")
print(f"2. Dataset File    : {data_path} ({len(df_raw):,} total rows)")
print(f"3. Active Window   : Trailing 90 days with 30d comparison windows")
print(f"4. Target Label    : Relative Impression Drop (< -15.0% observed drop)")
print(f"5. Excluded Fields : trend_pct, trend_direction, health_score (leakage & circular rules)")

=== DATA CONTRACT SPECIFICATION SUMMARY ===
1. Grain           : One row = 1 content_id (pseudonymized content page)
2. Dataset File    : ../../data/raw/content_refresh_anonymized.csv (30,000 total rows)
3. Active Window   : Trailing 90 days with 30d comparison windows
4. Target Label    : Relative Impression Drop (< -15.0% observed drop)
5. Excluded Fields : trend_pct, trend_direction, health_score (leakage & circular rules)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Categorization Bucket:

1. **Features (5 Maximum - Safe & Knowable):**
   - `ctr`: Knowable at decision moment because it records historical 90-day click-through rate prior to the prediction window.
   - `avg_position`: Knowable at decision moment because Google Search Console logs average search ranking position over the preceding 90 days.
   - `content_age_days`: Knowable at decision moment because publication timestamp is permanently recorded at content launch.
   - `days_since_last_update`: Knowable at decision moment because CMS editorial logs record the exact timestamp of the last update.
   - `engagement_rate`: Knowable at decision moment because GA4 user engagement metrics are compiled over the baseline 90-day window.

2. **Label / Proxy:**
   - `target_decay_flag`: Binary indicator ($\Delta \text{Imp}_{\text{rel}} < -15.0\%$) derived from observed search console telemetry.

3. **Context (Grouping & Splitting Only):**
   - `content_id`: Unique row identifier (never used as a numeric feature).
   - `client_id`: Pseudonymized client identifier (used strictly for `GroupKFold` validation splits).
   - `content_type`: Categorical content classification (e.g. `keyword article`).
   - `main_intent`: User intent classification (e.g. `informational`).

4. **Excluded (Prohibited Leakage & Product Flags):**
   - `trend_pct`: EXCLUDED — direct mathematical target leakage computed from the outcome window.
   - `trend_direction`: EXCLUDED — target-derived string representation of the outcome window.
   - `is_declining_label` / `health_score`: EXCLUDED — circular hand-written product flags.

In [2]:
# Section 2 Code: 5-Feature Frame Creation & Justification Verification

# Active Demand Slice (impressions_90d >= 100)
lane_slice = df_raw[df_raw['impressions_90d'] >= 100].copy().reset_index(drop=True)

# Derive observed target proxy
lane_slice['obs_imp_change_pct'] = (lane_slice['impressions_last_30d'] - lane_slice['impressions_prev_30d']) / (lane_slice['impressions_prev_30d'] + 1) * 100
lane_slice['target_decay_flag'] = (lane_slice['obs_imp_change_pct'] < -15.0).astype(int)

# Select 5 safe features max
safe_features = ['ctr', 'avg_position', 'content_age_days', 'days_since_last_update', 'engagement_rate']

feature_frame = lane_slice[['content_id', 'client_id'] + safe_features + ['target_decay_flag']].copy()

print("=== 5-FEATURE FRAME SUMMARY ===")
print(f"Feature Frame Shape: {feature_frame.shape}")
print("\nFeature Availability Justifications:")
justifications = {
    "ctr": "Knowable at decision moment: 90d historical CTR recorded in Search Console.",
    "avg_position": "Knowable at decision moment: 90d baseline search position log.",
    "content_age_days": "Knowable at decision moment: CMS publication timestamp.",
    "days_since_last_update": "Knowable at decision moment: CMS editorial revision log.",
    "engagement_rate": "Knowable at decision moment: GA4 user interaction telemetry."
}

for feat, just in justifications.items():
    print(f" - {feat:24s}: {just}")

print("\nFeature Frame Head:")
display(feature_frame.head(3))

=== 5-FEATURE FRAME SUMMARY ===
Feature Frame Shape: (22006, 8)

Feature Availability Justifications:
 - ctr                     : Knowable at decision moment: 90d historical CTR recorded in Search Console.
 - avg_position            : Knowable at decision moment: 90d baseline search position log.
 - content_age_days        : Knowable at decision moment: CMS publication timestamp.
 - days_since_last_update  : Knowable at decision moment: CMS editorial revision log.
 - engagement_rate         : Knowable at decision moment: GA4 user interaction telemetry.

Feature Frame Head:


,content_id,client_id,ctr,avg_position,content_age_days,days_since_last_update,engagement_rate,target_decay_flag
0,content_304f48230142,client_f369cb89fc,0.76,10.6,187,20,5.88,1
1,content_a1fb4e703a9e,client_4e07408562,0.05,20.3,445,25,0.00,1
2,content_9aa793d4d895,client_7f2253d7e2,0.09,36.5,141,20,0.00,1


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Three Verification Queries:

1. **Query 1 (Grain Probe Verification):** Check if `content_id` is strictly unique (`COUNT(*) > 1 HAVING` returns 0 rows).
2. **Query 2 (Row Count & Date Span):** Verify total raw inventory (30,000 rows), active slice count (22,006 rows), and content age range.
3. **Query 3 (Availability Filter with IS TRUE):** Filter active demand (`impressions_90d >= 100 IS TRUE`) to verify surviving row count (22,006 active demand pages / 73.35%).

### The Trap: Deliberate Feature Leakage Experiment
We intentionally introduce `trend_pct` into the feature set to observe fake perfection (Precision@50 = 100%), then remove it to preserve honest model benchmarks (Precision@50 = ~77.6%).

In [3]:
# Section 3 Code: Three Verification Queries & The Deliberate Leakage Trap

# --- QUERY 1: Grain Probe ---
grain_check = df_raw.groupby('content_id').size()
duplicates = grain_check[grain_check > 1]
print("=== QUERY 1: GRAIN VERIFICATION ===")
print(f"Duplicate content_ids found: {len(duplicates)} (Must be 0)")
assert len(duplicates) == 0, "Grain violation!"

# --- QUERY 2: Row Counts & Age Span ---
print("\n=== QUERY 2: ROW COUNTS & AGE SPAN ===")
print(f"Total Raw Rows       : {len(df_raw):,}")
print(f"Active Slice Rows    : {len(lane_slice):,} (impressions_90d >= 100)")
print(f"Unique Clients       : {lane_slice['client_id'].nunique()}")
print(f"Content Age Span     : {lane_slice['content_age_days'].min()} to {lane_slice['content_age_days'].max()} days")

# --- QUERY 3: Availability Filter with IS TRUE ---
availability_mask = (df_raw['impressions_90d'] >= 100) == True
surviving_count = availability_mask.sum()
print("\n=== QUERY 3: AVAILABILITY FILTER (IS TRUE) ===")
print(f"Surviving Rows (impressions_90d >= 100 IS TRUE): {surviving_count:,} / {len(df_raw):,} ({surviving_count/len(df_raw):.2%})")

# --- THE TRAP: Deliberate Leakage Experiment ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold

def evaluate_precision_at_50(y_true, y_prob, k=50):
    top_k_idx = np.argsort(y_prob)[::-1][:k]
    return np.mean(y_true.iloc[top_k_idx])

# A) Honest Model (5 Safe Features)
X_honest = lane_slice[safe_features].fillna(0)
y = lane_slice['target_decay_flag']
groups = lane_slice['client_id']

gkf = GroupKFold(n_splits=5)
honest_scores = []
for train_idx, test_idx in gkf.split(X_honest, y, groups):
    clf = RandomForestClassifier(n_estimators=30, random_state=42)
    clf.fit(X_honest.iloc[train_idx], y.iloc[train_idx])
    probs = clf.predict_proba(X_honest.iloc[test_idx])[:, 1]
    honest_scores.append(evaluate_precision_at_50(y.iloc[test_idx], probs, k=50))

honest_p50 = np.mean(honest_scores)

# B) LEAKED Model (Adding trend_pct deliberately)
X_leaked = X_honest.copy()
X_leaked['leaked_trend_pct'] = lane_slice['trend_pct'].fillna(0)

leaked_scores = []
for train_idx, test_idx in gkf.split(X_leaked, y, groups):
    clf = RandomForestClassifier(n_estimators=30, random_state=42)
    clf.fit(X_leaked.iloc[train_idx], y.iloc[train_idx])
    probs = clf.predict_proba(X_leaked.iloc[test_idx])[:, 1]
    leaked_scores.append(evaluate_precision_at_50(y.iloc[test_idx], probs, k=50))

leaked_p50 = np.mean(leaked_scores)

print("\n=== THE TRAP: DELIBERATE FEATURE LEAKAGE EXPERIMENT ===")
print(f"Honest Model Precision@50 (5 Safe Features) : {honest_p50:.2%}")
print(f"LEAKED Model Precision@50 (With trend_pct)  : {leaked_p50:.2%} (Fake 100% Perfection!)")
print("\nConclusion: The trap is sprung! Adding trend_pct gives a fake perfect score. We REMOVE trend_pct to maintain honest evaluation.")

=== QUERY 1: GRAIN VERIFICATION ===
Duplicate content_ids found: 0 (Must be 0)

=== QUERY 2: ROW COUNTS & AGE SPAN ===
Total Raw Rows       : 30,000
Active Slice Rows    : 22,006 (impressions_90d >= 100)
Unique Clients       : 30
Content Age Span     : 90 to 564 days

=== QUERY 3: AVAILABILITY FILTER (IS TRUE) ===
Surviving Rows (impressions_90d >= 100 IS TRUE): 22,006 / 30,000 (73.35%)



=== THE TRAP: DELIBERATE FEATURE LEAKAGE EXPERIMENT ===
Honest Model Precision@50 (5 Safe Features) : 78.40%
LEAKED Model Precision@50 (With trend_pct)  : 100.00% (Fake 100% Perfection!)

Conclusion: The trap is sprung! Adding trend_pct gives a fake perfect score. We REMOVE trend_pct to maintain honest evaluation.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named Slice Limitations:

1. **Unbalanced Client History & GA4 Tracking Gaps:** 
   - Historical depth varies significantly across clients. In the warehouse release, rows preceding a client's `ga4_data_start` have GA4 columns zero-filled with `ga4_data_available = FALSE`. Filtering on `ga4_data_available = TRUE` is required to prevent treating missing telemetry as zero engagement.
2. **Static Snapshot Seasonality Blindspot:** 
   - A single 90-day snapshot cannot separate macro seasonal demand drops (e.g. holiday search drops) from genuine content quality decay unless impressions are normalized against position-tier CTR expectations.
3. **Query Table 90-Day Overlap:** 
   - In the warehouse query table (`fact_content_query_90d`), context metrics repeat per query row and overlap trailing snapshot windows. Aggregating query terms requires `ANY_VALUE()` rather than `SUM()` to prevent double-counting.

In [4]:
# Section 4 Code: Data Limits & Panel Integrity Audit
missingness_summary = df_raw[safe_features].isnull().mean() * 100

print("=== DATA LIMITS & FEATURE MISSINGNESS AUDIT ===")
print("Feature Missingness Rates (%):")
for feat, rate in missingness_summary.items():
    print(f" - {feat:24s}: {rate:.2f}% missing")

print("\nNamed Limitation Statement:")
print("Static snapshots cannot distinguish seasonal search volume drops from content decay without multi-month panel tracking and position-adjusted CTR baselines.")

=== DATA LIMITS & FEATURE MISSINGNESS AUDIT ===
Feature Missingness Rates (%):
 - ctr                     : 0.00% missing
 - avg_position            : 0.00% missing
 - content_age_days        : 0.00% missing
 - days_since_last_update  : 0.00% missing
 - engagement_rate         : 0.00% missing

Named Limitation Statement:
Static snapshots cannot distinguish seasonal search volume drops from content decay without multi-month panel tracking and position-adjusted CTR baselines.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.